# 15 打开 minGPT，把每个组件认出来

## 1 在真实项目里认出自己写过的计算

上一课已经生成了一段文本。现在打开 minGPT，目标是看懂：同样的 QKV、残差和下一字符预测，在一个真实项目里怎样组织。遇到更短、更快的写法时，可以拿自己的逐头实现逐项对照。

`mingpt/model.py` 定义注意力、Block 和 GPT；`trainer.py` 接训练循环；`projects/chargpt` 将文本变成窗口。本地副本在 `references/minGPT`，下面先列出模型中的类与前向函数，找到熟悉的入口。

In [1]:
from support import ROOT, check, show, style

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
torch.set_num_threads(1)
torch.manual_seed(7)
style()

In [3]:
reference=ROOT/'references'/'minGPT'
model_source=(reference/'mingpt/model.py').read_text(encoding='utf-8')
for number,line in enumerate(model_source.splitlines(),1):
    if line.startswith('class ') or 'def forward(' in line:
        print(number,line.strip())

21 class NewGELU(nn.Module):
26 def forward(self, x):
29 class CausalSelfAttention(nn.Module):
52 def forward(self, x):
73 class Block(nn.Module):
90 def forward(self, x):
95 class GPT(nn.Module):
260 def forward(self, idx, targets=None):


## 2 用带编号的张量理解多头换轴

原项目把所有头的投影合在一次 Linear 中。投影后的 C=H×D 先拆成 `(B,T,H,D)`，再交换 T、H 成 `(B,H,T,D)`。矩阵乘法会分别处理每个 B、H。

下面填入编号，不进行训练，只检查换轴前后“哪个数属于哪个位置、哪一头”。

In [4]:
B,T,H,D=1,3,2,2
numbered=torch.arange(B*T*H*D).reshape(B,T,H,D)
heads=numbered.transpose(1,2)
print('原本B,T,H,D：\n',numbered)
print('变成B,H,T,D：\n',heads)
restored=heads.transpose(1,2).contiguous().view(B,T,H*D)
check('换回来没有丢数',restored,numbered.reshape(B,T,H*D))

原本B,T,H,D：
 tensor([[[[ 0,  1],
          [ 2,  3]],

         [[ 4,  5],
          [ 6,  7]],

         [[ 8,  9],
          [10, 11]]]])
变成B,H,T,D：
 tensor([[[[ 0,  1],
          [ 4,  5],
          [ 8,  9]],

         [[ 2,  3],
          [ 6,  7],
          [10, 11]]]])
通过：换回来没有丢数


### 练习 m01：取第二头的第二个位置

从 heads 取 batch 0、head 1、time 1 的向量。

In [5]:
ex_m01_vector=None

In [6]:
# 填写上一格后，运行检查。
check('头与位置对应',ex_m01_vector,[6,7])

待填写：头与位置对应


<details>
<summary>提示</summary>

heads 的轴顺序已经是 B,H,T,D。

</details>

<details>
<summary>参考答案</summary>

```python
ex_m01_vector=heads[0,1,1]
```

</details>

`contiguous()` 按新的逻辑顺序准备连续存储，接着 `view(B,T,C)` 合并头与头内特征。`transpose` 改的是轴的解释，不能用直接 reshape 来替代这次换轴。

### 练习 m02：合并所有头

把 heads 换回 B,T,H,D，再合并最后两个轴。

In [7]:
ex_m02_joined=None

In [8]:
# 填写上一格后，运行检查。
check('合并头',ex_m02_joined,restored)

待填写：合并头


<details>
<summary>提示</summary>

先 transpose(1,2)，再 contiguous().view(...)。

</details>

<details>
<summary>参考答案</summary>

```python
ex_m02_joined=heads.transpose(1,2).contiguous().view(B,T,H*D)
```

</details>

## 3 用原项目创建一个小模型

配置里选择一种方式指定结构：这里直接给层数、头数、宽度，所以 model_type 设为 None。将 Dropout 设为 0，以便做数值对照。

In [9]:
import sys
sys.path.insert(0,str(reference))
from mingpt.model import GPT
configuration=GPT.get_default_config()
configuration.model_type=None
configuration.n_layer,configuration.n_head,configuration.n_embd=1,2,16
configuration.vocab_size,configuration.block_size=7,8
configuration.embd_pdrop=configuration.resid_pdrop=configuration.attn_pdrop=0.
upstream=GPT(configuration)

number of parameters: 0.00M


In [10]:
input_ids=torch.tensor([[0,1,2,3],[3,2,1,0]])
target_ids=torch.tensor([[1,2,3,4],[2,1,0,1]])
upstream_logits,upstream_loss=upstream(input_ids,target_ids)
print('输出：',upstream_logits.shape,'loss：',upstream_loss.item())
check('原项目交叉熵',upstream_loss,F.cross_entropy(upstream_logits.reshape(-1,7),target_ids.reshape(-1)))

输出： torch.Size([2, 4, 7]) loss： 1.9429577589035034
通过：原项目交叉熵


### 练习 m03：从 logits 得到概率

对 upstream_logits 的最后一个轴做 Softmax。

In [11]:
ex_m03_probs=None

In [12]:
# 填写上一格后，运行检查。
check('每位置概率和',None if ex_m03_probs is None else ex_m03_probs.sum(dim=-1),torch.ones(2,4))

待填写：每位置概率和


<details>
<summary>提示</summary>

最后一轴是词表。

</details>

<details>
<summary>参考答案</summary>

```python
ex_m03_probs=torch.softmax(upstream_logits,dim=-1)
```

</details>

## 4 从结果判断模型有没有读到未来

三角形 mask 写在源码里，还可以从模型的实际输出验证它。下面只改变最后一个输入位置；第0–2位置面对的过去没有变，它们的输出也应相同。这个小干预把“因果”变成一个可以观察的结果。

In [13]:
upstream.eval()
original,_=upstream(input_ids)
perturbed=input_ids.clone(); perturbed[:,-1]=6
after,_=upstream(perturbed)
check('真实minGPT的因果性',after[:,:-1],original[:,:-1])

通过：真实minGPT的因果性


### 练习 m04：挑选可比较的位置

比较最后一个输入之前的全部 logits。返回它们的最大绝对差。

In [14]:
ex_m04_difference=None

In [15]:
# 填写上一格后，运行检查。
check('前缀分数不受未来影响',ex_m04_difference,0.)

待填写：前缀分数不受未来影响


<details>
<summary>提示</summary>

用 after[:,:-1]-original[:,:-1]，再 abs().max()。

</details>

<details>
<summary>参考答案</summary>

```python
ex_m04_difference=(after[:,:-1]-original[:,:-1]).abs().max()
```

</details>

## 5 打开训练器后，自己写出它的一步

原项目 Trainer 的核心仍是前向、清梯度、反传、更新。这里用同一小 batch 重复 20 步，观察实际损失；原项目还在这几步外围添加 DataLoader、回调和梯度裁剪。

In [16]:
upstream.train()
upstream_optimizer=torch.optim.AdamW(upstream.parameters(),lr=.01)
upstream_history=[]
for step in range(20):
    _,loss=upstream(input_ids,target_ids)
    upstream_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    upstream_optimizer.step()
    upstream_history.append(loss.item())
print('同一batch的loss：',upstream_history[0],'->',upstream_history[-1])

同一batch的loss： 1.9429577589035034 -> 0.17514720559120178


### 练习 m05：重新算训练目标损失

用原项目返回的 logits 自己计算一次平均交叉熵。

In [17]:
ex_m05_loss=None

In [18]:
# 填写上一格后，运行检查。
check('交叉熵接口一致',ex_m05_loss,upstream_loss)

待填写：交叉熵接口一致


<details>
<summary>提示</summary>

合并batch、位置，保留7个词表分数。

</details>

<details>
<summary>参考答案</summary>

```python
ex_m05_loss=F.cross_entropy(upstream_logits.reshape(-1,7),target_ids.reshape(-1))
```

</details>

## 6 观察中间层，再提出可检验的问题

Forward hook 在指定模块计算后保存输出。先核对这个输出的形状；随后才能研究某层保留了什么信息。hook 的三项参数是模块、输入元组、输出。

In [19]:
cache={}
def record_hidden(module,inputs,output):
    cache['hidden']=output.detach()
handle=upstream.transformer.h[0].register_forward_hook(record_hidden)
upstream.eval()
_ = upstream(input_ids)
handle.remove()
print('Block输出：',cache['hidden'].shape)

Block输出： torch.Size([2, 4, 16])


### 练习 m06：把隐藏状态投影到词表

使用最终归一化与输出投影，把 cache 中的状态转成词表分数。

In [20]:
ex_m06_lens=None

In [21]:
# 填写上一格后，运行检查。
check('隐藏状态投影形状',None if ex_m06_lens is None else ex_m06_lens.shape,(2,4,7))

待填写：隐藏状态投影形状


<details>
<summary>提示</summary>

依次使用 upstream.transformer.ln_f 和 upstream.lm_head。

</details>

<details>
<summary>参考答案</summary>

```python
ex_m06_lens=upstream.lm_head(upstream.transformer.ln_f(cache['hidden']))
```

</details>

## 接到 NTU 作业

HW01 的 token、prompt 和多轮对话建立在模型接口之上；先用本课的整数输入把接口走通。HW03 的 Logit Lens 与 Function Vector 涉及中间表示和干预：可先在第14课训练的模型上记录隐藏状态，固定输入、只改变一层的某个向量，再观察损失变化。

独立任务：在 minGPT 找出 mask、LayerNorm、两次残差、目标错位、训练器更新五处源码；每处写一段数值检查。只训练同一小batch测的是拟合能力；再构造独立序列检验泛化。

## 带着问题读原文

[minGPT 原项目](https://github.com/karpathy/minGPT)

依次读 model.py 的 CausalSelfAttention、Block、GPT.forward，再读 trainer.py 的 run。比较这里逐头实现与原项目合并投影实现，哪些是数学结构，哪些是计算效率选择？